# Frozen GTEx fusion reference: baseline and repeat coverage

This Stage 3 notebook establishes the local IsoFormer-style frozen pilot and keeps its later repeat-seed comparison alongside that method. One transcript supplies genomic DNA context, RNA, protein and 30 tissue RNA-abundance targets. The pilot combines unequal-length frozen representations with trained cross-attention and a prediction head; it is not a full reproduction of the published end-to-end experiment.

The baseline section retains its original seed-0 preparation, resource check, fits and evidence. The repeat section retains its separate reading/computation switches, requested seeds and result directory. Historical outputs remain with their original cells; the new arrangement has not been executed.

The [shared pilot implementation](../shared_code/fusion/isoformer.py) and [synthetic checks](../../tests/test_isoformer.py) define this computation. [LucaOne's pooled comparison](../stage1/lucaone_representation_comparison.ipynb) is a separate representation study. [Fuller released-reference reproduction remains deferred](../stage4/RUNBOOK.md#isoformer-reproduction-specification). Gantt/Details task 44 retains the pilot and reference context.


---

## Setup

Use the [repository environment](../../README.md#notebook-dependencies) and start the kernel from `Code/stage3/` to retain the baseline's existing relative cache and result destinations. Reading retained outputs needs no execution. Running the baseline section can read the pilot CSV, extract embeddings, probe a model, fit models and recalculate metrics; its original execution behavior remains unchanged.

The later repeat section has a separate setup. Its `READ_SAVED_RESULTS = True` skips repeat cache access and fitting; that flag does not disable the preceding baseline section. To read repeat coverage alone, execute these imports/shared-code setup cells and the repeat section, skipping the baseline computation. Both original workflows keep their own settings and device preferences.


In [ ]:
# Standard-library paths, environment settings and timing.
import json
import os
import sys
import time
from pathlib import Path

# Enable supported Apple MPS CPU fallbacks before importing PyTorch.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Arrays and tables for sequence lengths, tissue targets and result summaries.
import numpy as np
import pandas as pd
from IPython.display import display

# Tensor operations, the memory probe and hardware availability.
import torch


In [1]:
# Locate the shared project code while retaining the repository-root fallback.
CODE_DIR = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "shared_code").is_dir()),
    Path.cwd() / "Code",
)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from shared_code import datasets, encoders
from shared_code.fusion import isoformer, isoformer_results, run_identity, train
from shared_code.fusion import token_embeddings

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("torch", torch.__version__, "| Apple GPU available:", torch.backends.mps.is_available())


torch 2.10.0+cu126 | Apple GPU available: False


---

## Establish the frozen GTEx pilot

One GTEx row describes a transcript isoform through genomic DNA context, its RNA sequence and its protein sequence, with **30 tissue RNA-abundance targets**. The tissue columns are outputs to predict, not additional input modalities. They are not direct measures of translation efficiency or protein production. The [GTEx target guide](../../docs/methods/data-and-targets.md#gtex-tissue-expression-labels) explains the supplied log-transformed values and unresolved normalization details.

The next table preserves the [IsoFormer paper's Table 2](../../Papers/core/Multi-Modal-Transfer-Learning.pdf), an NT-based modality comparison. It motivates asking which inputs help in this pilot. The published fine-tuned experiment and our frozen pilot are separate configurations.


In [4]:
# Published NT-based modality ablation (Table 2), distinct from this frozen pilot.
# The stored paper values report tissue-averaged metrics and variability over five seeds.
ISOFORMER_TABLE2 = pd.DataFrame(
    [
        ("dna", 0.13, 0.02, 0.43, 0.01),
        ("rna", 0.36, 0.03, 0.61, 0.01),
        ("prot", 0.20, 0.01, 0.46, 0.01),
        ("dna+prot", 0.28, 0.01, 0.52, 0.01),
        ("dna+rna", 0.39, 0.01, 0.64, 0.01),
        ("dna+rna+prot", 0.43, 0.01, 0.65, 0.01),
    ],
    columns=["variant", "R2", "R2 sd", "Spearman", "Spearman sd"],
).set_index("variant")
display(ISOFORMER_TABLE2)
published_best_single = ISOFORMER_TABLE2.loc[["dna", "rna", "prot"], "Spearman"].max()
published_fusion_gain = ISOFORMER_TABLE2.loc["dna+rna+prot", "Spearman"] - published_best_single
print(
    f"published fusion gain in Spearman: {published_fusion_gain:+.2f}, alongside a marginal seed SD of "
    f"{ISOFORMER_TABLE2['Spearman sd'].max():.2f}"
)


,R2,R2 sd,Spearman,Spearman sd
variant,,,,
dna,0.13,0.02,0.43,0.01
rna,0.36,0.03,0.61,0.01
prot,0.20,0.01,0.46,0.01
dna+prot,0.28,0.01,0.52,0.01
dna+rna,0.39,0.01,0.64,0.01
dna+rna+prot,0.43,0.01,0.65,0.01


their fusion gain in Spearman: +0.04, against a seed sd of 0.01


The published means and seed standard deviations describe performance and variation across the paper's repeated fits. Comparing a fusion gain with one model's marginal standard deviation is descriptive; it is not a paired test of the gain. Likewise, the modality ordering is a question to investigate here, not a result that must survive changes to the experiment.

**What the pilot changes.** The [pilot builder](../stage1/build_gtex_pilot.py) samples the release's supplied training and test partitions. The retained pilot has 2,000 rows, initially split 1,000/1,000; the loader reserves 5% of its training rows for validation, leaving 950 for fitting in the saved run. That inner holdout samples transcript rows rather than gene groups. It does not independently certify gene-family or species generalization.

The pilot freezes NT v2 100M for DNA/RNA and ESM-2 150M for protein. Its tissue targets remain as supplied, without applying the release's additional normalization statistics. Cohort size, NT checkpoint, context, training and aggregation choices also differ from the paper. Therefore, the score gap **does not measure the cost of freezing**. The [paper/release review](../../docs/sources.md#isoformer-dataset-and-model-cards) records those distinctions and the released Enformer-based configuration.

### Pilot settings

The settings preserve the original crops, six variants, seed and training cap. Batches are constrained by both row count and an estimated attention-cell budget. This groups similarly costly transcripts to reduce padding overhead; it is not a guarantee about peak device memory. The hardware preference remains MPS, then CUDA, then CPU.


In [ ]:
BENCHMARK = "gtex_pilot"
# Keep paths relative to the kernel working directory, as in the original runs.
CACHE_ROOT = Path("embeddings")
RESULTS_DIR = Path("results") / BENCHMARK
# Only legacy pilot caches consult this separate reconstruction; fresh caches use their metadata.
GTEX_CACHE_RECOVERY = CODE_DIR / "stage3" / "gtex_cache_recovery.json"

# Batch rows are capped by both count and the estimated attention-cell budget.
MAX_BATCH = 16
CELL_BUDGET = 300_000_000
VAL_FRACTION = 0.05  # shared loader randomly holds out transcript rows from training
MAX_EPOCHS = 500  # existing pilot cap, not a recovered paper stopping rule
N_HEADS = 8
SEEDS = (0,)  # selected pilot seed; additional repeats remain separate executions

# DNA is unchanged; RNA keeps its 3' suffix; protein keeps its N-terminal prefix.
CROP = {"dna": None, "rna": 12_000, "prot": 1_022}
# Retained content-token budgets; the preliminary length estimate is not tokenization.
CONTEXT = {"ntv2_100m": 2_048, "esm2_150m": 1_022}
VARIANTS = ["dna", "rna", "prot", "dna+prot", "dna+rna", "dna+rna+prot"]


In [ ]:
# The pilot is independent of the former LucaOne feasibility section.
gtex = pd.read_csv(datasets.DEFAULT_DATA_DIR / isoformer.GTEX_PILOT)


In [5]:
# Preserve the pilot's MPS -> CUDA -> CPU preference.
DEVICE = (
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(
    f"{len(gtex):,} transcripts | splits {dict(gtex['split'].value_counts())} | "
    f"{len(datasets.GTEX_TISSUES)} tissues | device {DEVICE}"
)
print("encoder roles:", {m: isoformer.STREAM_ENCODER[m] for m in isoformer.MODALITIES})


2,000 transcripts | splits {'train': np.int64(1000), 'test': np.int64(1000)} | 30 tissues | device cuda
encoder roles: {'dna': 'ntv2_100m', 'rna': 'ntv2_100m', 'prot': 'esm2_150m'}


### Prepare and cache the three inputs

DNA uses the supplied genomic window, RNA retains its 3′ suffix when cropped, and protein retains its N-terminal prefix. The helper then applies the selected encoder's input formatting. These GTEx caches are separate from the BLF caches in [step 2](blf_token_embeddings.ipynb): DNA and RNA share an NT checkpoint here but have different input strings, so cache names include the stream.

The preliminary token estimate uses sequence length and the retained sixmer arithmetic. Ambiguous bases and k-mer remainders make this different from actually tokenizing every input; extraction checks the actual tokenized inputs. Cache reuse also requires compatible evidence, not merely existing shards. Fresh caches use their own input metadata. Complete historical pilot caches use the separate [GTEx recovery record](gtex_cache_recovery.json), which verifies the recorded files and reconstructed input association without rewriting them. Missing or mismatched legacy shards stop this path; they are not silently regenerated. This remains qualified historical reuse, not proof of the original producing execution.


In [ ]:
# Prepare one stream and count affected rows; no encoder is loaded by this helper.
def stream_texts(modality):
    column = isoformer.STREAM_COLUMN[modality]
    encoder = encoders.ENCODERS[isoformer.STREAM_ENCODER[modality]]
    raw_sequences = gtex[column].astype(str).str.strip().str.upper()
    crop_limit = CROP[modality]
    if crop_limit is None:
        kept_sequences = raw_sequences
    elif modality == "rna":
        kept_sequences = raw_sequences.str[-crop_limit:]  # retain the pilot's RNA 3' suffix
    else:
        kept_sequences = raw_sequences.str[:crop_limit]
    cropped_count = int((raw_sequences.str.len() > (crop_limit or 10**9)).sum())
    input_texts = [
        (
            encoders.protein_input(encoder, sequence)
            if encoder.modality == "protein"
            else encoders.encoder_input(encoder, sequence)
        )
        for sequence in kept_sequences
    ]
    return input_texts, cropped_count


In [ ]:
def release_pilot_resources(streams, primary_error=None):
    """Release mappings and unused allocator memory without hiding the original failure."""
    failure = primary_error
    for cleanup in (streams.close, encoders.free_memory):
        try:
            cleanup()
        except BaseException as error:
            if failure is None:
                failure = error
            elif hasattr(failure, "add_note"):
                failure.add_note(f"Pilot cleanup also failed: {error!r}")
    if primary_error is None and failure is not None:
        raise failure


In [ ]:
def prepare_pilot_caches():
    """Extract new inputs or verify complete old pilot caches; return one owned stream set."""
    caches = {}
    streams = isoformer.GTExStreams(
        BENCHMARK, gtex, caches, tuple(datasets.GTEX_TISSUES),
    )
    verification = token_embeddings.CacheVerification()
    try:
        for modality in isoformer.MODALITIES:
            input_texts, cropped_count = stream_texts(modality)
            encoder_key = isoformer.STREAM_ENCODER[modality]
            cache_name = f"{modality}_{encoder_key}"
            directory = CACHE_ROOT / BENCHMARK / cache_name
            longest_input_length = max(len(text.replace(" ", "")) for text in input_texts)
            # Retain the preliminary sixmer estimate; extraction checks actual tokenization.
            estimated_content_tokens = (
                longest_input_length
                if encoders.ENCODERS[encoder_key].modality == "protein"
                else longest_input_length // 6 + longest_input_length % 6
            )
            if estimated_content_tokens > CONTEXT[encoder_key]:
                raise ValueError(
                    f"{cache_name}: longest input is {estimated_content_tokens:,} tokens, over the "
                    f"{CONTEXT[encoder_key]:,} the encoder accepts; lower CROP"
                )
            print(
                f"  longest-length estimate {estimated_content_tokens:,} content tokens "
                f"against a {CONTEXT[encoder_key]:,} budget"
            )
            print(
                f"\n{cache_name}: {len(input_texts):,} rows, "
                f"{cropped_count:,} cropped to {CROP[modality]}"
            )
            metadata_path = directory / "meta.json"
            metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}
            if not isinstance(metadata, dict):
                raise ValueError(f"{metadata_path}: metadata must be an object.")
            legacy = bool(metadata) and not (
                "input_sha256" in metadata or "shards" in metadata
            )
            if legacy:
                # Summarize only: an absent legacy shard must fail, never trigger extraction.
                cache = token_embeddings.summarize(directory)
                status = "historical cache; verification follows"
            else:
                cache, computed = token_embeddings.extract(
                    encoders.ENCODERS[encoder_key], input_texts, directory, DEVICE,
                    verification=verification,
                )
                status = "extracted" if computed else "already complete"
            caches[modality] = cache
            print(
                f"  {status}: {len(cache):,} rows, width {cache.width}, "
                f"{cache.meta['total_tokens']:,} tokens"
            )
        streams.check()
        # This read-only route verifies modern manifests or the exact legacy reconstruction.
        run_identity.gtex_cache_records(
            streams, crop=CROP, verification=verification,
            gtex_recovery=GTEX_CACHE_RECOVERY,
        )
        return streams
    except BaseException as error:
        release_pilot_resources(streams, primary_error=error)
        raise


In [6]:
# Preparation owns the caches until it returns; the probe and each fit release their mappings.
STREAMS = prepare_pilot_caches()


  longest input 1,000 tokens against a 2,048 budget

dna_ntv2_100m: 2,000 rows, 0 cropped to None


W1022 04:27:47.011000 32468 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


    ntv2_100m: shard 1/8 (1/8 this run)
    ntv2_100m: shard 2/8 (2/8 this run)
    ntv2_100m: shard 3/8 (3/8 this run)
    ntv2_100m: shard 4/8 (4/8 this run)
    ntv2_100m: shard 5/8 (5/8 this run)
    ntv2_100m: shard 6/8 (6/8 this run)
    ntv2_100m: shard 7/8 (7/8 this run)
    ntv2_100m: shard 8/8 (8/8 this run)
  extracted: 2,000 rows, width 512, 2,000,000 tokens
  longest input 2,000 tokens against a 2,048 budget

rna_ntv2_100m: 2,000 rows, 11 cropped to 12000
    ntv2_100m: shard 1/8 (1/8 this run)
    ntv2_100m: shard 2/8 (2/8 this run)
    ntv2_100m: shard 3/8 (3/8 this run)
    ntv2_100m: shard 4/8 (4/8 this run)
    ntv2_100m: shard 5/8 (5/8 this run)
    ntv2_100m: shard 6/8 (6/8 this run)
    ntv2_100m: shard 7/8 (7/8 this run)
    ntv2_100m: shard 8/8 (8/8 this run)
  extracted: 2,000 rows, width 512, 776,882 tokens
  longest input 1,022 tokens against a 1,022 budget

prot_esm2_150m: 2,000 rows, 124 cropped to 1022


Loading weights:   0%|          | 0/486 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    esm2_150m: shard 1/8 (1/8 this run)
    esm2_150m: shard 2/8 (2/8 this run)
    esm2_150m: shard 3/8 (3/8 this run)
    esm2_150m: shard 4/8 (4/8 this run)
    esm2_150m: shard 5/8 (5/8 this run)
    esm2_150m: shard 6/8 (6/8 this run)
    esm2_150m: shard 7/8 (7/8 this run)
    esm2_150m: shard 8/8 (8/8 this run)
  extracted: 2,000 rows, width 640, 696,364 tokens


### Construct the six modality comparisons

The local [aggregator](../shared_code/fusion/isoformer.py) updates each present stream by attending successively to the other streams, using their original key/value representations within that layer. Residual updates and LayerNorm preserve each query stream's width; masks exclude padded positions. Each updated stream is then mean-pooled, the pooled vectors are concatenated, and a linear head predicts all 30 tissue targets. With one stream, the model reduces to masked pooling plus that head.

This is the pilot's computation. It differs from the released Enformer-based path that updates and pools DNA before its prediction head; agreement with the exact historical NT implementation is not established by that comparison. Fits materialize only their selected streams. All three streams' length metadata still determine the original batch membership and ordering, including for single-modality fits, so this loading optimization does not change the pilot's minibatches.

### Inspect the batches before fitting

The loader derives validation only from the pilot's training partition and leaves its supplied test rows untouched. The same seed controls the transcript holdout and batch ordering. The next cells report the actual loader sizes, rank batches by padded attention cost, and probe the largest estimated batch.

The cost sums query-length × context-length across ordered modality pairs, then multiplies by attention heads and batch size. Multiplying by four gives a hypothetical float32 attention-map payload, **not total GPU memory**: other activations, gradients and workspaces matter, and an attention kernel may not materialize every map. The probe runs a forward/backward pass without an optimizer step or explicit accelerator synchronization. Its epoch extrapolation is a rough diagnostic, not a measured training-time estimate.


In [ ]:
# Reuse the verified stream set; validation comes from training and test rows stay fixed.
LOADERS = isoformer.loaders_for(
    STREAMS, batch_size=MAX_BATCH, val_fraction=VAL_FRACTION, budget=CELL_BUDGET, n_heads=N_HEADS
)
for name, loader in LOADERS.items():
    batch_sizes = [len(b) for b in loader.batch_sampler.batches]
    print(
        f"{name:<6} {len(loader.dataset):>5} rows in {len(batch_sizes):>3} batches, "
        f"size {min(batch_sizes)} to {max(batch_sizes)}"
    )
print("widths", STREAMS.widths)


In [ ]:
# Rank batches by the existing padded pairwise-attention proxy, not measured peak memory.
batch_costs = []
for position, batch in enumerate(LOADERS["train"].batch_sampler.batches):
    row_indices = LOADERS["train"].dataset.indices[batch]
    max_lengths = {
        m: int(np.asarray(STREAMS.caches[m].lengths)[row_indices].max())
        for m in isoformer.MODALITIES
    }
    batch_costs.append(
        (
            len(batch)
            * N_HEADS
            * sum(
                max_lengths[a] * max_lengths[b] for a in max_lengths for b in max_lengths if a != b
            ),
            position,
            max_lengths,
        )
    )
largest_attention_cost, largest_batch_position, largest_batch_lengths = max(batch_costs)
print(
    f"heaviest of {len(batch_costs)} training batches: {largest_batch_lengths}, "
    f"{largest_attention_cost / 1e6:,.0f}M attention cells, {largest_attention_cost * 4 / 1e9:.2f} GB if every attention cell were stored as float32"
)


In [ ]:
def probe_largest_batch(streams, loaders, batch_position, batch_count, device, n_heads):
    """Run the original forward/backward probe and release its tensors on every exit."""
    model = batch = prediction = diagnostics = None
    primary_error = None
    try:
        model = isoformer.builder(streams, "dna+rna+prot", None, n_heads=n_heads)
        model = model.to(device)
        batch = isoformer.collate([
            loaders["train"].dataset[i]
            for i in loaders["train"].batch_sampler.batches[batch_position]
        ])
        started = time.perf_counter()
        prediction, diagnostics = model(batch.to(device))
        torch.nn.MSELoss()(prediction, batch.labels.to(device)).backward()
        seconds = time.perf_counter() - started
        print(
            f"probe completed in {seconds:.1f}s; largest-batch extrapolation "
            f"{seconds * batch_count / 60:.1f} min/epoch (unsynchronized diagnostic, "
            "not a training-time estimate)"
        )
    except BaseException as error:
        primary_error = error
        if isinstance(error, (RuntimeError, MemoryError)):
            print(f"probe failed: {type(error).__name__}: {str(error)[:140]}")
            print("If this is an allocation failure, lower CELL_BUDGET before rebuilding loaders.")
        raise
    finally:
        # Backward frees saved intermediates; drop remaining graph/output references as well.
        model = batch = prediction = diagnostics = None
        release_pilot_resources(streams, primary_error=primary_error)


In [7]:
# Diagnostic only: forward/backward on the largest estimated batch, with no optimizer step.
probe_largest_batch(
    STREAMS, LOADERS, largest_batch_position, len(batch_costs), DEVICE, N_HEADS,
)


val       50 rows in   7 batches, size 1 to 16
train    950 rows in  91 batches, size 2 to 16
test    1000 rows in  96 batches, size 3 to 16
widths {'dna': 512, 'rna': 512, 'prot': 640}
heaviest of 91 training batches: {'dna': 1000, 'rna': 947, 'prot': 578}, 298M attention cells, 1.19 GB of maps
the heaviest batch fits: 0.8s, so roughly 1.1 min per epoch for the three-modality model


### Fit the selected variants

Each seed repeats the six variants in the declared order. The shared result tag is used for both fitting and reading so the budgeted cohort remains distinct from earlier filenames. The execution adapter preserves compatible completed results under its existing checks; it does not reconstruct missing historical producer metadata or make an incompatible checkpoint resumable.

Passing the memory probe does not guarantee all later batches and training allocations fit. Each fit loads only its selected streams, preserves the original batch schedule, and releases mapped shards on success or failure. New fits use verified cache identity; the recovery record qualifies only the retained historical pilot caches.


In [ ]:
def pilot_result_tag(variant, seed):
    """Use the same existing filename for fitting and reading each pilot result."""
    return (
        f"{BENCHMARK}__{variant}__heads{N_HEADS}" f"__budget{CELL_BUDGET // 1_000_000}M__seed{seed}"
    )


In [ ]:
def fit_pilot_variant(streams, variant, seed, verification):
    """Request one fit with the retained schedule and release its cache mappings afterward."""
    variant_loaders = None
    primary_error = None
    try:
        variant_loaders = isoformer.loaders_for(
            streams, batch_size=MAX_BATCH, val_fraction=VAL_FRACTION, seed=seed,
            budget=CELL_BUDGET, n_heads=N_HEADS,
            # Only materialization changes; all-three length metadata still defines batches.
            modalities=tuple(variant.split("+")),
        )
        config = train.TrainConfig(batch_size=MAX_BATCH, max_epochs=MAX_EPOCHS, seed=seed)
        tag = pilot_result_tag(variant, seed)
        execution_records = run_identity.execution_records(
            streams, variant, config, RESULTS_DIR, tag=tag, loaders=variant_loaders,
            model_kwargs={"n_heads": N_HEADS}, crop=CROP, verification=verification,
            repository=CODE_DIR.parent, device=DEVICE, gtex_recovery=GTEX_CACHE_RECOVERY,
        )
        train.run(
            streams, variant, config, DEVICE, RESULTS_DIR, builder=isoformer.builder,
            loaders=variant_loaders, tag=tag, n_heads=N_HEADS, **execution_records,
        )
    except BaseException as error:
        primary_error = error
        raise
    finally:
        variant_loaders = None
        release_pilot_resources(streams, primary_error=primary_error)


In [ ]:
# Preserve seed/variant order. Compatible completed results are reused without replacement.
RUN_CACHE_VERIFICATION = token_embeddings.CacheVerification()
for seed in SEEDS:
    print(f"\n--- seed {seed} ---")
    for variant in VARIANTS:
        fit_pilot_variant(STREAMS, variant, seed, RUN_CACHE_VERIFICATION)



--- seed 0 ---
  gtex_pilot__dna__heads8__budget300M__seed0: training
    epoch   0  train loss 11.9132  val 0.0171  best 0.0171 @ 0
    epoch  10  train loss 5.8640  val 0.1014  best 0.1014 @ 10
    epoch  20  train loss 5.4083  val 0.1206  best 0.1206 @ 20
    epoch  30  train loss 5.5036  val 0.1406  best 0.1406 @ 30
    epoch  40  train loss 5.5117  val 0.1600  best 0.1600 @ 40
    epoch  50  train loss 5.4998  val 0.1776  best 0.1776 @ 50
    epoch  60  train loss 5.4868  val 0.1912  best 0.1912 @ 60
    epoch  70  train loss 5.4685  val 0.2040  best 0.2040 @ 70
    epoch  80  train loss 5.4463  val 0.2146  best 0.2146 @ 80
    epoch  90  train loss 5.4428  val 0.2214  best 0.2214 @ 90
    epoch 100  train loss 5.4262  val 0.2279  best 0.2279 @ 100
    epoch 110  train loss 5.4140  val 0.2297  best 0.2304 @ 109
    epoch 120  train loss 5.4038  val 0.2316  best 0.2316 @ 120
    epoch 130  train loss 5.3913  val 0.2337  best 0.2337 @ 130
    epoch 140  train loss 5.3871  val 0.232

### Read the scores and examine the ordering

The reader checks the selected budgeted recipe, reports missing or incompatible files, and requires every requested seed before summarizing a variant. It validates finite prediction/target panels and matching target order. Spearman is read from each result; $R^2$ is recalculated from its validated saved predictions. Both metrics are averaged across tissues; Spearman measures transcript ranking within each tissue, while $R^2$ compares squared prediction error with the error of predicting that tissue's mean.

Historical seed-0 records still have limited identity: modality labels, attention heads and budget rely on the documented filenames/caller, and agreement of target values does not prove the original embedding inputs. Modern identities must validate when present. The ordering checks require all named candidates on the requested seed set; they are descriptive, not significance tests or a certification of replication. Reaching the epoch cap alone does not establish continued validation improvement.


In [ ]:
def summarize_pilot_results(cohort):
    """Summarize only variants with every requested seed and validated predictions."""
    complete = {
        row["variant"] for row in cohort.coverage if row["status"] == "complete"
    }
    comparison_rows = []
    for variant in VARIANTS:
        if variant not in complete:
            continue
        # Keep declared seed order, including the original averaging order.
        records = {row["seed"]: row["result"] for row in cohort.records
                   if row["variant"] == variant}
        results = [records[seed] for seed in SEEDS]
        spearman_scores = [result["test_score"] for result in results]
        r2_scores = [
            train.r_squared(np.asarray(result["test_truth"]), np.asarray(result["test_prediction"]))
            for result in results
        ]
        epochs_completed = [result["epochs"] for result in results]
        comparison_rows.append({
            "variant": variant,
            "seeds": len(results),
            "our Spearman": np.mean(spearman_scores),
            "our R2": np.mean(r2_scores),
            "their Spearman": ISOFORMER_TABLE2.loc[variant, "Spearman"],
            "their R2": ISOFORMER_TABLE2.loc[variant, "R2"],
            "epochs": int(np.mean(epochs_completed)),
            "hit the cap": "yes" if max(epochs_completed) >= MAX_EPOCHS else "",
        })
    columns = [
        "variant", "seeds", "our Spearman", "our R2", "their Spearman", "their R2",
        "epochs", "hit the cap",
    ]
    return pd.DataFrame(comparison_rows, columns=columns).set_index("variant")


In [ ]:
pilot_cohort = isoformer_results.read_cohort(
    RESULTS_DIR,
    train_config=vars(train.TrainConfig(batch_size=MAX_BATCH, max_epochs=MAX_EPOCHS)),
    variants=VARIANTS, expected_seeds=SEEDS,
    n_heads=N_HEADS, budget=CELL_BUDGET, val_fraction=VAL_FRACTION,
    require_predictions=True,
    expected_shape=(int((gtex["split"] == "test").sum()), len(datasets.GTEX_TISSUES)),
)
display(pd.DataFrame(pilot_cohort.coverage).set_index("variant"))
if pilot_cohort.issues:
    display(pd.DataFrame(pilot_cohort.issues))
comparison = summarize_pilot_results(pilot_cohort)


In [ ]:
display(comparison.round(3))
if comparison.empty:
    print("No complete compatible variants; see coverage above. No replacement fits were requested.")
if (comparison["hit the cap"] == "yes").any():
    capped = list(comparison.index[comparison["hit the cap"] == "yes"])
    print(
        f"NOTE: {', '.join(capped)} reached MAX_EPOCHS. The cap alone does not establish\n"
        "continued improvement or bound a fusion gain; inspect the validation history."
    )


In [8]:
# Descriptive orderings from the available pilot scores, not significance tests.
# Each displayed variant has the same complete requested seed set.
pilot_scores = comparison["our Spearman"].to_dict()
available_singles = [
    variant_name for variant_name in ("dna", "rna", "prot") if variant_name in pilot_scores
]
ordering_checks = []
if len(available_singles) == 3:
    ordering_checks.append(
        {
            "claim": "RNA is the strongest single modality",
            "result": (
                "ok"
                if max(available_singles, key=lambda variant_name: pilot_scores[variant_name])
                == "rna"
                else "NO"
            ),
            "detail": ", ".join(
                f"{variant_name} {pilot_scores[variant_name]:.3f}"
                for variant_name in available_singles
            ),
        }
    )
if "dna+rna+prot" in pilot_scores and len(available_singles) == 3:
    best_available_single_score = max(
        pilot_scores[variant_name] for variant_name in available_singles
    )
    ordering_checks.append(
        {
            "claim": "all three modalities beat the best single one",
            "result": "ok" if pilot_scores["dna+rna+prot"] > best_available_single_score else "NO",
            "detail": f"three {pilot_scores['dna+rna+prot']:.3f} against best single {best_available_single_score:.3f}",
        }
    )
if "dna+rna" in pilot_scores and "dna+prot" in pilot_scores:
    ordering_checks.append(
        {
            "claim": "DNA with RNA beats DNA with protein",
            "result": "ok" if pilot_scores["dna+rna"] > pilot_scores["dna+prot"] else "NO",
            "detail": f"dna+rna {pilot_scores['dna+rna']:.3f} against "
            f"dna+prot {pilot_scores['dna+prot']:.3f}",
        }
    )
if ordering_checks:
    display(pd.DataFrame(ordering_checks).set_index("claim"))


,seeds,our Spearman,our R2,their Spearman,their R2,epochs,hit the cap
variant,,,,,,,
dna,1,0.199,0.015,0.43,0.13,154,
rna,1,0.452,0.205,0.61,0.36,359,
prot,1,0.160,0.009,0.46,0.20,65,
dna+prot,1,0.439,0.182,0.52,0.28,32,
dna+rna,1,0.507,0.243,0.64,0.39,40,
dna+rna+prot,1,0.499,0.231,0.65,0.43,26,


,result,detail
claim,,
RNA is the strongest single modality,ok,"dna 0.199, rna 0.452, prot 0.160"
all three modalities beat the best single one,ok,three 0.499 against best single 0.452
DNA with RNA beats DNA with protein,ok,dna+rna 0.507 against dna+prot 0.439


## Deviations recorded for this step

These are the pilot's implemented choices and comparison limits. They are not all verified reconstructions of the authors' historical NT experiment; the [source review](../../docs/sources.md#isoformer-dataset-and-model-cards) distinguishes paper descriptions, released code and local assumptions.

| Choice | Pilot treatment | Interpretation limit |
| --- | --- | --- |
| Encoder adaptation | Frozen token representations; only aggregation/head parameters are trained | A difference from the fine-tuned paper, confounded with the other changes below |
| Backbones and context | NT v2 100M for DNA/RNA, ESM-2 150M for protein; retained DNA/RNA/protein crops | Different capacity and available context; Enformer is a separate reference |
| Cohort and targets | Pilot sampled from supplied partitions; targets copied unchanged | Smaller cohort and no additional release normalization; inner validation is not gene-grouped |
| Aggregation | One layer; each stream queries the others in fixed order | A local all-stream adaptation, not the released DNA-query path |
| Normalization and pooling | LayerNorm after residual updates; masked pooling of each stream | These choices belong to the fitted block and cannot be attributed to attention alone |
| Prediction and loss | Concatenated pooled vectors, linear tissue head and mean squared error | Head capacity and target scale affect comparison; the release uses a different head |
| Batching | Attention-cell budget with at most 16 rows | Changes batch membership and size; the budget is not measured peak memory |
| Training | Existing shared trainer and 500-epoch cap, one selected seed | Does not recover the full paper recipe or measure robustness across initialization |

## What the reference establishes

The retained pilot scores rank RNA highest among the single inputs, and DNA+RNA above RNA alone. The [recorded paired analysis](../../docs/run-records.md#2026-10-19-stage3-isoformer-gtex-isoformers-aggregator-on-the-gtex-pilot-frozen-encoders) supports a gain for DNA+RNA on that one-seed test comparison without resolving further benefit from protein. This notebook itself does not calculate those bootstrap intervals. They do not establish a biological mechanism or robustness across new fits.

Together with the BLF build, this provides another inspectable fusion computation and a task where the input streams carry different sequence context. Comparing GTEx results with fungal BLF results changes the task and data as well as the architecture. A direct test of fusion or alignment would need matched inputs, evaluation and suitable single/concatenation controls.

The repeat-coverage section below extends this same reference method. The [Stage 4 investigation](../stage4/README.md) owns proposed controlled comparisons and the [LucaOne notebook](../stage1/lucaone_representation_comparison.ipynb) keeps its stability findings separate. The pilot's existence does not establish those proposed outcomes or make a full published reproduction a prerequisite.


---

## Check the same pilot across additional seeds

The baseline above establishes the local attention method. This section asks how much the local GTEx modality comparisons vary when the cross-attention predictor is trained again. One transcript record supplies a genomic DNA window, transcript RNA, a protein sequence and expression targets for 30 tissues. The tissue values are outputs to predict; the modality variants select which sequence inputs are available.

Those targets describe tissue-averaged transcript abundance from bulk RNA sequencing, supplied on the release's stated `log(x + 1)` scale. Our pilot copies these values directly and does not apply the separate normalization statistics supplied for the original IsoFormer procedure. The local predictor trains with mean squared error on these stored labels; its main score averages Spearman correlations across transcripts within each tissue. It does not score the ordering of tissues within one transcript or a percentage gain in expression. The [GTEx target guide](../../docs/methods/data-and-targets.md#gtex-tissue-expression-labels) explains the measurement and remaining provenance limits.

The local IsoFormer-style model combines unequal-length token representations through cross-attention, without BioLangFusion's codon grid. It updates each available stream, then averages and concatenates the updated streams for prediction. The [released Enformer-based implementation](https://huggingface.co/InstaDeepAI/isoformer/blob/afac2a386c8f2241faf8fad3c5971a9307c13775/modeling_isoformer.py) instead updates DNA using RNA and then protein, pools DNA, and applies its expression head. The local helper therefore studies an adaptation of the method. Its encoders remain frozen. [Stage 3's modality comparison](../../docs/run-records.md#2026-10-19-stage3-isoformer-gtex-isoformers-aggregator-on-the-gtex-pilot-frozen-encoders) provides the seed-0 evidence. This block requests seeds 1 and 2 for the same pilot. Changing the seed changes initialization and the 5% validation subset drawn from training data; the test partition remains fixed.

Three intended local seeds differ from the paper's five-seed, end-to-end fine-tuning procedure. The earlier “one hour on CUDA/four on Mac” estimate lacks sufficient device and run detail to serve as a forecast. Runtime depends on sequence lengths, batch budget, hardware and training duration.

### Configure the additional runs

The separate repeat setup below owns its execution switches and paths. `READ_SAVED_RESULTS` applies only to this repeat section; the baseline section retains its own original execution behavior. The prefixed settings below retain the existing modality combinations and training limits. A helper defines one variant/seed fit; the following cell opens the streams and requests those fits in the retained seed-then-variant order. `REPEAT_CELL_BUDGET` limits estimated attention cells in a batch; it is not a byte or GPU-memory limit. `REPEAT_MAX_EPOCHS` is a ceiling, with the shared trainer's stopping rules still applying.

The [GTEx recovery record](../stage3/gtex_cache_recovery.json) permits qualified reuse of the inspected pilot matrices. It reconstructs their input association from the unchanged CSV, extraction recipe and stored lengths, and identifies the current files with checksums. It does not recover an extraction-time manifest or prove that the original seed-0 fits used byte-identical matrices. Cache selection is automatic: fresh caches use their own input and shard metadata; `REPEAT_CACHE_RECOVERY` supplies a fallback only for the exact inspected old files that lack both fields. Invalid or mismatched modern metadata remains an error. Setting the path to `None` disables that fallback. Reading saved results needs neither option nor cache access, and the training switches remain separate.


In [ ]:
# Independent repeat workflow: keep its original read mode, paths and CUDA preference.
READ_SAVED_RESULTS = True
RUN_ISOFORMER_SEEDS = False
STAGE3_DIR = CODE_DIR / "stage3"
STAGE4_DIR = CODE_DIR / "stage4"
REPEAT_CACHE_ROOT = STAGE3_DIR / "embeddings"
REPEAT_RESULTS_DIR = STAGE4_DIR / "results" / "gtex_isoformer"
REPEAT_CACHE_RECOVERY = STAGE3_DIR / "gtex_cache_recovery.json"
tokens = token_embeddings
REPEAT_DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


In [ ]:
ISOFORMER_SEEDS = (1, 2)
REPEAT_VARIANTS = ("dna", "rna", "prot", "dna+prot", "dna+rna", "dna+rna+prot")

REPEAT_MAX_BATCH = 16              # Upper bound on examples per batch.
REPEAT_CELL_BUDGET = 300_000_000   # Estimated attention cells, not bytes.
REPEAT_N_HEADS = 8                 # Cross-attention heads.
REPEAT_VAL_FRACTION = 0.05         # Fraction of the training partition used for validation.
REPEAT_MAX_EPOCHS = 500            # Maximum training epochs per fit.


In [ ]:
def fit_isoformer_variant(gtex_streams, variant, seed, verification):
    """Prepare and request one retained pilot fit; the caller owns the streams."""
    made = isoformer.loaders_for(
        gtex_streams,
        batch_size=REPEAT_MAX_BATCH,
        val_fraction=REPEAT_VAL_FRACTION,
        seed=seed,
        budget=REPEAT_CELL_BUDGET,
        n_heads=REPEAT_N_HEADS,
    )
    settings = train.TrainConfig(
        batch_size=REPEAT_MAX_BATCH, max_epochs=REPEAT_MAX_EPOCHS, seed=seed,
    )
    tag = (f"gtex_pilot__{variant}__heads{REPEAT_N_HEADS}"
           f"__budget{REPEAT_CELL_BUDGET // 1_000_000}M__seed{seed}")
    records = run_identity.execution_records(
        gtex_streams, variant, settings, REPEAT_RESULTS_DIR, tag=tag, loaders=made,
        model_kwargs={"n_heads": REPEAT_N_HEADS}, verification=verification,
        repository=CODE_DIR.parent, device=REPEAT_DEVICE,
        gtex_recovery=REPEAT_CACHE_RECOVERY,
    )
    train.run(
        gtex_streams,
        variant,
        settings,
        REPEAT_DEVICE,
        REPEAT_RESULTS_DIR,
        builder=isoformer.builder,
        loaders=made,
        tag=tag,
        n_heads=REPEAT_N_HEADS,
        **records,
    )


In [20]:
if READ_SAVED_RESULTS:
    print("Saved-results mode: IsoFormer cache opening and seed fits skipped.")
else:
    if RUN_ISOFORMER_SEEDS:
        run_verification = tokens.CacheVerification()
        gtex_streams = isoformer.GTExStreams.open(REPEAT_CACHE_ROOT)
        for seed in ISOFORMER_SEEDS:
            print(f"\n--- seed {seed} ---")
            for variant in REPEAT_VARIANTS:
                fit_isoformer_variant(gtex_streams, variant, seed, run_verification)
    else:
        print("RUN_ISOFORMER_SEEDS is off; the reading cell below works with whatever seeds exist.")


RUN_ISOFORMER_SEEDS is off; the reading cell below works with whatever seeds exist.


### Read the reference values and local results

The reference table retains IsoFormer's published Table 2 Spearman values for each modality combination. These are context for a local frozen-encoder pilot, not target values that would establish reproduction. A score gap can reflect the cohort, inputs, encoders, architecture, normalization and training together; it cannot identify the effect of freezing alone. The [Stage 4 guide](README.md#gtex-input-suitability-and-a-frozen-fusion-reference) explains the comparison boundary.

The [cohort reader](../shared_code/fusion/isoformer_results.py) keeps the six modality combinations separate. It accepts the six known baseline seed-0 reference filenames with heads8 and budget300M, mapping their historical generic IsoFormer label to the modality in each exact filename. Other reference configurations are excluded, and a current record's stored modality must agree with its filename. Reading does not change those files.

The stored training configuration must match this notebook's settings. Attention heads and loader budget are selected from filenames; validation fraction, input identity and historical code/model settings are not established by the saved records. These limits remain visible in the diagnostics, so a matching filename is not treated as complete provenance.

When new fits use the recovered caches alongside the historical seed-0 scores, interpret the combined table as a qualified pilot comparison. Its spread is not demonstrated to isolate training-seed variation on identical historical embeddings; the earlier fit record and later extraction record describe different execution environments. The recovery status stays in each new fit's identity rather than being added to old results.


In [ ]:
REPEAT_REFERENCE_TABLE = pd.DataFrame(
    [
        ("dna", 0.43),
        ("rna", 0.61),
        ("prot", 0.46),
        ("dna+prot", 0.52),
        ("dna+rna", 0.64),
        ("dna+rna+prot", 0.65),
    ],
    columns=["variant", "their Spearman"],
).set_index("variant")


In [ ]:
# Match the retained seed-0 reference and the requested additional seeds.
# This reads saved JSON metadata and scores; it does not load encoders or train.
isoformer_cohort = isoformer_results.read_cohort(
    STAGE3_DIR / "results" / "gtex_pilot",
    REPEAT_RESULTS_DIR,
    train_config=vars(train.TrainConfig(
        batch_size=REPEAT_MAX_BATCH, max_epochs=REPEAT_MAX_EPOCHS,
    )),
    variants=REPEAT_VARIANTS,
    expected_seeds=(0, *ISOFORMER_SEEDS),
    n_heads=REPEAT_N_HEADS,
    budget=REPEAT_CELL_BUDGET,
    val_fraction=REPEAT_VAL_FRACTION,
)
found = isoformer_cohort.records


### Check the selected records and seed coverage

For each modality combination, check the actual available seeds against seed 0 and the requested additional seeds. The diagnostics identify missing destinations, excluded configurations, ambiguous duplicate keys and the limits of historical metadata. Duplicate modality/seed identities block the summary instead of silently keeping the first file.

The mean and standard deviation below summarize only the selected observations. Their count can be smaller than the intended seed count; one available seed has no estimated standard deviation. An excluded or missing run is not a zero score, and these standard deviations are not bootstrap confidence intervals.


In [ ]:
coverage = pd.DataFrame(isoformer_cohort.coverage).set_index("variant")
display(coverage)
if isoformer_cohort.issues:
    display(pd.DataFrame(isoformer_cohort.issues))


In [21]:
if isoformer_cohort.duplicate_keys:
    if READ_SAVED_RESULTS:
        print("Duplicate modality/seed records shown above; summary skipped.")
    else:
        raise ValueError(
            "Resolve the duplicate modality/seed records shown above before summarizing."
        )
elif found:
    # Each included row is one distinct modality/seed observation.
    runs = pd.DataFrame(found)
    table = (
        runs.groupby("variant")["Spearman"]
        .agg(["mean", "std", "count"])
        .reindex(REPEAT_VARIANTS)
    )
    table["count"] = table["count"].fillna(0).astype(int)
    display(table.join(REPEAT_REFERENCE_TABLE).round(3))
    if table["count"].max() > 1:
        print(f"seed spread, largest standard deviation: {table['std'].max():.3f}")
else:
    print("No compatible IsoFormer runs found; inspect the coverage and exclusions above.")


,mean,std,count,their Spearman
variant,,,,
IsoFormer,0.439,NaN,1,NaN


---

## Reference coverage and later experiments

Comparing variation across IsoFormer-style GTEx runs requires correctly identified and compatible records for each modality combination and seed. The coverage table reports available and missing seeds for each combination; duplicate identities block the summary. A partial mean is based only on the displayed count, and one seed does not establish a seed spread. Matching stored settings and filename tags still leaves the historical provenance limits described above. Consult the individual [Stage 3 result record](../../docs/run-records.md#2026-10-19-stage3-isoformer-gtex-isoformers-aggregator-on-the-gtex-pilot-frozen-encoders) and the [Stage 4 runbook](../stage4/RUNBOOK.md#the-order) when deciding which evidence can be reused.

A focused next question could ask whether the local attention block improves prediction over pooling and concatenating the same frozen inputs, with rows, partitions and evaluation held fixed. That contrast would test the whole block, including its normalization and additional parameters. It is a proposed comparison, not a result or an execution path implemented by this notebook. The [runbook](../stage4/RUNBOOK.md) retains the method-study direction and the deferred fuller-reproduction specification. No full replication is a prerequisite for understanding these results.
